# 07 · Query-side LLM work (vLLM)

Plan sections 4.4–4.5; pipeline step 3. For all 100 questions:

* **Q1 HyDE**: one hypothetical CS/AMC-style passage per question;
* **Q2 keyword expansion**: up to 10 regulation-style terms (JSON);
* **F1 filter extraction**: `{engine_type: "piston" | "turbine" | null}` (JSON).

Qwen3.8-27B-INT4, thinking off, temperature 0, JSON-schema output where JSON is needed. Every call is cached in `data/cache/llm.jsonl` under the hash of the full request, so notebook 08 reads these outputs without calling the model. The outputs don't depend on the index.

**Before running:** vLLM with the retrieval config, which stays up for notebooks 07–08:
```bash
VLLM_CONFIG=serve.retrieval.yaml docker compose up -d vllm   # or set VLLM_CONFIG in .env
docker compose logs -f vllm                                   # ~2.5 min to healthy
```

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # all paths below are relative to the repo root
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [ ]:
import json, re, subprocess, time, urllib.request
import pandas as pd
from tqdm.auto import tqdm

base = os.environ.get("VLLM_ENDPOINT", "http://127.0.0.1:8000/v1")
health = base.rsplit("/v1", 1)[0] + "/health"
t0 = time.time()
while True:
    try:
        urllib.request.urlopen(health, timeout=5)
        break
    except Exception:
        if time.time() - t0 > 900:
            raise TimeoutError(f"{health} not healthy; check `docker compose logs vllm`")
        time.sleep(10)
print("vLLM healthy after", round(time.time() - t0), "s")

### KV-cache check (plan 3.1)

The startup log must show a KV cache that holds at least one full 32,768-token request. If it doesn't, set `max-model-len: 24576` in `vllm/serve.retrieval.yaml` and drop the 16k expansion budget.

In [ ]:
try:
    log = subprocess.run(["docker", "logs", "ikincitik-vllm"], capture_output=True, text=True, timeout=60)
    text = log.stdout + log.stderr
    conc = [float(x) for x in re.findall(r"Maximum concurrency for 32,768 tokens per request: ([\d.]+)x", text)]
    kv = re.findall(r"GPU KV cache size: ([\d,]+) tokens", text)
    print("KV cache tokens:", kv[-1] if kv else "?", "| max concurrency at 32768:", conc[-1] if conc else "?")
    assert conc and conc[-1] >= 1.0, "KV cache does not hold one 32768-token request"
except FileNotFoundError:
    print("docker CLI not available here: check `docker compose logs vllm | grep concurrency` by hand")

In [ ]:
from csrag.runs import Caches
from csrag.retrievers.query_llm import LLM, prompt_hashes

questions = [json.loads(l) for l in open(DATA / "eval" / "questions.jsonl", encoding="utf-8")]
llm = LLM(Caches(DATA / "cache").llm)
print(llm.model, "| prompts:", prompt_hashes())

In [ ]:
rows = []
for q in tqdm(questions):
    passage, r1 = llm.hyde(q["question"])
    terms, r2 = llm.keywords(q["question"])
    engine, r3 = llm.extract_filter(q["question"])
    rows.append({"question_id": q["question_id"], "question": q["question"], "hyde": passage, "terms": terms,
                 "f1_engine_type": engine, "label_engine_type": q["engine_type"], "answerable": q["answerable"],
                 "llm_seconds": r1["seconds"] + r2["seconds"] + r3["seconds"],
                 "prompt_tokens": sum(r["usage"]["prompt_tokens"] for r in (r1, r2, r3)),
                 "completion_tokens": sum(r["usage"]["completion_tokens"] for r in (r1, r2, r3)),
                 "finish": [r["finish_reason"] for r in (r1, r2, r3)]})
out = pd.DataFrame(rows)
print(len(llm.cache), "cached LLM calls")
out[["llm_seconds", "prompt_tokens", "completion_tokens"]].describe().loc[["mean", "50%", "max"]]

In [ ]:
truncated = out[out["finish"].apply(lambda f: "length" in f)]
print(len(truncated), "questions with an output cut at max_tokens")
pd.set_option("display.max_colwidth", 200)
out[["question_id", "question", "hyde", "terms"]].head(5)

### F1 extraction vs the `engine_type` label (evaluation only)

F1 never sees the label. The label is only compared here, and the oracle F0 is the one method that uses it. `all` questions should map to `null` (no filter); a wrong `piston`/`turbine` filter removes the gold section from every list.

In [ ]:
pd.crosstab(out["label_engine_type"], out["f1_engine_type"].fillna("null (no filter)"), margins=True)